###### Imports and Settings

In [10]:
import pandas as pd
import geopandas as gpd
import numpy as np
import requests
import io
import pickle
import matplotlib.pyplot as plt
from collections import deque
from functools import reduce
%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

###### Functions

In [11]:
#function for percent of whole
def percent(x, y):
    try:
        return round((x/y)*100, 2)
    except ZeroDivisionError:
        return 666
#function for population density
def populationdensity(x, y):
    try:
        return round(x/y, 2)
    except ZeroDivisionError:
        return 666

# Comprehensive Plan Data Pull  

The following API calls are designed to streamline the data pulls for the comprehensive plans for any geography. For the sake of simplicity, there are three separate documents for ACS variables, Decennial 2000 SF3 variables, and Decennial SF1 variables. This document contains ACS 5-Year Estimate variables.

In [12]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)

In [13]:
#create a variable that contains your api key
api_key = keys_dict_2['CENSUS']
#api_key = '24fc7d81b74510d599f702dbd408fb18e1466d81'

In [14]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149', #Rutherford
       '119'] #Maury

## Read In Data Guide

The "head" should never be more than 2 variables, and the "tail" never more than 2. Since we can pull 50 variables at once this means that we can systematically program in 46 variables for each pull, so that's:
+ dg1: ID's 1 - 46  
+ dg2: ID's 47-92  
+ dg3: ID's 93-138  
+ dg4: ID's 139-184  
+ dg5: ID's 185-230  
+ dg6: ID's 231-276  
+ dg7: ID's 277-322  
+ dg8: ID's 323-368  
+ dg9: ID's 369-414  
+ dg10: ID's 415-460  
+ dg11: ID's 461-506  
+ dg12: ID's 507-552  
+ dg13: ID's 553-598  
+ dg14: ID's 599-644  
+ dg15: ID's 645-690  
+ dg16: ID's 691-736  
+ dg17: ID's 737-782  
+ dg18: ID's 783-828  
**This data guide stops at ID 783 as of 5/18/2022.**

In [15]:
dataguide = pd.read_csv('../data guides/DATA GUIDE ACS.csv', dtype = str)
dataguide['ID'] = dataguide['ID'].astype(int)

In [20]:
dg1 = dataguide[dataguide['ID'].between(1, 46)]
dg2 = dataguide[dataguide['ID'].between(47, 92)]
dg3 = dataguide[dataguide['ID'].between(93, 138)]
dg4 = dataguide[dataguide['ID'].between(139, 184)]
dg5 = dataguide[dataguide['ID'].between(185, 230)]
dg6 = dataguide[dataguide['ID'].between(231, 276)]
dg7 = dataguide[dataguide['ID'].between(277, 322)]
dg8 = dataguide[dataguide['ID'].between(323, 368)]
dg9 = dataguide[dataguide['ID'].between(369, 414)]
dg10 = dataguide[dataguide['ID'].between(415, 460)]
dg11 = dataguide[dataguide['ID'].between(461, 506)]
dg12 = dataguide[dataguide['ID'].between(507, 552)]
dg13 = dataguide[dataguide['ID'].between(553, 598)]
dg14 = dataguide[dataguide['ID'].between(599, 644)]
dg15 = dataguide[dataguide['ID'].between(645, 690)]
dg16 = dataguide[dataguide['ID'].between(691, 736)]
dg17 = dataguide[dataguide['ID'].between(737, 782)]
dg18 = dataguide[dataguide['ID'].between(783, 828)]

## One Through Eighteen

In [124]:
# ONE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg1['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg1['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
one = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
one = one.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
one = one.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [125]:
# TWO
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg2['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg2['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
two = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
two = two.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
two = two.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [126]:
# THREE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg3['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg3['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
three = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
three = three.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
three = three.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [127]:
# FOUR
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg4['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg4['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
four = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
four = four.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
four = four.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [128]:
# FIVE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg5['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg5['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
five = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
five = five.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
five = five.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [129]:
# SIX
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg6['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg6['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
six = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
six = six.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
six = six.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [130]:
# SEVEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg7['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg7['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
seven = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
seven = seven.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
seven = seven.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [131]:
# EIGHT
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg8['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg8['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eight = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
eight = eight.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eight = eight.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [132]:
# NINE
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg9['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg9['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
nine = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
nine = nine.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
nine = nine.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [133]:
# TEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg10['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg10['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
ten = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
ten = ten.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
ten = ten.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [134]:
# ELEVEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg11['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg11['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eleven = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
eleven = eleven.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eleven = eleven.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [83]:
state

,NAME,GEO_ID,fb_am_la_centr_costarica,fb_am_la_centr_elsalvador,fb_am_la_centr_guatemala,fb_am_la_centr_honduras,fb_am_la_centr_mexico,fb_am_la_centr_nicaragua,fb_am_la_centr_panama,fb_am_la_centr_other,fb_am_la_southamerica,fb_am_la_s_argentina,fb_am_la_s_bolivia,fb_am_la_s_brazil,fb_am_la_s_chile,fb_am_la_s_columbia,fb_am_la_s_ecuador,fb_am_la_s_guyana,fb_am_la_s_peru,fb_am_la_s_uruguay,fb_am_la_s_venezuela,fb_am_la_s_other,fb_am_northern,fb_am_n_canada,fb_am_n_other,language_total_5+_series,language_total_5to7,language_5to7_onlyenglish,language_5to7_spanish,language_5to7_otherindoeuropean,language_5to7_asianandpacificisland,language_5to7_other,language_total_18to64,language_18to64_onlyenglish,language_18to64_spanish,language_18to64_otherindoeuropean,language_18to64_asianandpacificisland,language_18to64_other,language_total_65+,language_65+_onlyenglish,language_65+_spanish,language_65+_otherindoeuropean,language_65+_asianandpacificisland,language_65+_other,attainment_total_over25_series,attainment_noschooling,attainment_nurseryschool,attainment_kindergarten,StateFIPS
0,Tennessee,0400000US47,727,9272,15359,12374,89257,638,960,41,15255,1149,302,2192,830,3249,886,426,2581,167,3355,118,8146,8004,142,6365282,1100904,997301,71768,11974,9584,10277,4156814,3829675,181173,57854,52235,35877,1107564,1078655,11593,8627,7226,1463,4649847,49000,652,594,47


In [81]:
state

,NAME,GEO_ID,attainment_1stgrade,attainment_2ndgrade,attainment_3rdgrade,attainment_4thgrade,attainment_5thgrade,attainment_6thgrade,attainment_7thgrade,attainment_8thgrade,attainment_9thgrade,attainment_10thgrade,attainment_11thgrade,attainment_12thgradenodiploma,attainment_regularhighschooldiploma,attainment_gedoralternativecredential,attainment_somecollegelessthan1year,attainment_somecollege1ormoreyearsnodegree,attainment_associatesdegree,attainment_bachelorsdegree,attainment_mastersdegree,attainment_professionalschooldegree,attainment_doctoratedegree,classworker_total_series,classworker_privateforprofit,classworker_privateforprofit_employee,classworker_privateforprofit_selfemployedincorporatedbusiness,classworker_privatenotforprofit,classworker_localgovt,classworker_stategovt,classworker_federalgovt,classworker_selfemployednonincorporatedbusiness,classworker_unpaidfamilyworker,lfstatus_total_sexbyagebyemploymentstatus16+_series,lfstatus_mciv_16to19,lfstatus_mciv_16to19_employed,lfstatus_mciv_16to19_unemployed,lfstatus_mciv_20to21,lfstatus_mciv_20to21_employed,lfstatus_mciv_20to21_unemployed,lfstatus_mciv_22to24,lfstatus_mciv_20to21_employed,lfstatus_mciv_20to21_unemployed,lfstatus_mciv_25to29,lfstatus_mciv_25to29_employed,lfstatus_mciv_25to29_unemployed,lfstatus_mciv_30to34,lfstatus_mciv_30to34_employed,StateFIPS
0,Tennessee,0400000US47,1592,2026,6697,5267,8835,26412,21193,74241,70688,98443,103597,78496,1214151,265497,301940,662392,349162,826947,339015,83857,59153,3103430,2227649,2151346,76303,235284,218717,118814,91289,206196,5481,5437242,68931,57154,11777,65992,58219,7773,105788,96520,9268,205463,191728,13735,187383,178283,47


In [115]:
# Twelve
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg12['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg12['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
twelve = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
twelve = twelve.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
twelve = twelve.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [116]:
# THIRTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg13['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg13['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
thirteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
thirteen = thirteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
thirteen = thirteen.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [117]:
# FOURTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg14['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg14['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
fourteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
fourteen = fourteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
fourteen = fourteen.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [118]:
# FIFTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg15['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg15['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
fifteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
thirteen = fifteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
fifteen = fifteen.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [119]:
# SIXTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg16['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg16['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
sixteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
thirteen = sixteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
sixteen = sixteen.append(state, ignore_index= True)
print('Okay Finished')

Okay Finished


In [120]:
# SEVENTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg17['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg17['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
seventeen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
seventeen = seventeen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
seventeen = seventeen.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [121]:
# EIGHTEEN
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg18['ACS Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg18['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
data_appended = []
for i in GNRC:
    url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
    predicates= {}
    get_vars= var_list
    predicates["get"]= ",". join(get_vars)
    predicates["for"]= "county:{}".format(i)
    predicates["in"]= "state:47"
    data= requests.get(url_str, params= predicates)
    col_names = col_list
    data=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
    data_appended.append(data)
data_appended = pd.concat(data_appended).reset_index(drop = True)
eighteen = data_appended
#places call
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[(places['GEO_ID']=='1600000US4722360')|(places['GEO_ID']=='1600000US4751560')|(places['GEO_ID']=='1600000US4741200')
                  |(places['GEO_ID']=='1600000US4769420')|(places['GEO_ID']=='1600000US4778320')|(places['GEO_ID']=='1600000US4750780')
                  |(places['GEO_ID']=='1600000US4741520')]
eighteen = eighteen.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2020/acs/acs5?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47".format(i)
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
eighteen = eighteen.append(state, ignore_index = True)
print('Okay Finished')

Okay Finished


In [135]:
dfs = [two,three,four,five,six,seven,eight,nine,ten,eleven,twelve,thirteen,fourteen,fifteen,sixteen,seventeen]

In [136]:
for df in dfs:
    df = df.drop(columns = ['NAME','StateFIPS','GeoFIPS'], inplace = True)

In [137]:
one = one.drop(columns = ['StateFIPS','GeoFIPS'])
eighteen = eighteen.drop(columns = 'NAME')

The default inplace=False is intended for assigning back to the original dataframe, because it returns a new copy. However inplace=True operates on the same copy and returns None, therefore there is no need to assign back to the original dataframe.

In [138]:
df_merged = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)

In [139]:
df_merged.head()

,GEO_ID,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,healthcoverage_m55to64_wprivate,healthcoverage_m55to64_woprivate,healthcoverage_m65to74_privateall,healthcoverage_m65to74_wprivate,healthcoverage_m65to74_woprivate,healthcoverage_m75+_privateall,healthcoverage_m75+_wprivate,healthcoverage_m75+_woprivate,healthcoverage_total_fprivate,healthcoverage_fu6_privateall,healthcoverage_fu6_wprivate,healthcoverage_fu6_woprivate,healthcoverage_f6to18_privateall,healthcoverage_f6to18_wprivate,healthcoverage_f6to18_woprivate,healthcoverage_f19to25_privateall,healthcoverage_f19to25_wprivate,healthcoverage_f19to25_woprivate,healthcoverage_f26to34_privateall,healthcoverage_f26to34_wprivate,healthcoverage_f26to34_woprivate,healthcoverage_f35to44_privateall,healthcoverage_f35to44_wprivate,healthcoverage_f35to44_woprivate,healthcoverage_f45to54_privateall,healthcoverage_f45to54_wprivate,healthcoverage_f45to54_woprivate,healthcoverage_f55to64_privateall,healthcoverage_f55to64_wprivate,healthcoverage_f55to64_woprivate,healthcoverage_f65to74_privateall,healthcoverage_f65to74_wprivate,healthcoverage_f65to74_woprivate,healthcoverage_f75+_privateall,healthcoverage_f75+_wprivate,healthcoverage_f75+_woprivate,healthcoverage_total_publichealthcare_series,healthcoverage_total_mpublic,healthcoverage_mu6_publicall,healthcoverage_mu6_wpublic,healthcoverage_mu6_wopublic,healthcoverage_m6to18_publicall,healthcoverage_m6to18_wpublic,healthcoverage_m6to18_wopublic,healthcoverage_m19to25_publicall,healthcoverage_m19to25_wpublic,healthcoverage_m19to25_wopublic,healt

In [140]:
dfs = [one, df_merged, eighteen]
data = reduce(lambda  left,right: pd.merge(left,right,on=['GEO_ID'],
                                            how='outer'), dfs)

In [141]:
data.head()

,NAME,GEO_ID,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,healthcoverage_m55to64_wprivate,healthcoverage_m55to64_woprivate,healthcoverage_m65to74_privateall,healthcoverage_m65to74_wprivate,healthcoverage_m65to74_woprivate,healthcoverage_m75+_privateall,healthcoverage_m75+_wprivate,healthcoverage_m75+_woprivate,healthcoverage_total_fprivate,healthcoverage_fu6_privateall,healthcoverage_fu6_wprivate,healthcoverage_fu6_woprivate,healthcoverage_f6to18_privateall,healthcoverage_f6to18_wprivate,healthcoverage_f6to18_woprivate,healthcoverage_f19to25_privateall,healthcoverage_f19to25_wprivate,healthcoverage_f19to25_woprivate,healthcoverage_f26to34_privateall,healthcoverage_f26to34_wprivate,healthcoverage_f26to34_woprivate,healthcoverage_f35to44_privateall,healthcoverage_f35to44_wprivate,healthcoverage_f35to44_woprivate,healthcoverage_f45to54_privateall,healthcoverage_f45to54_wprivate,healthcoverage_f45to54_woprivate,healthcoverage_f55to64_privateall,healthcoverage_f55t

In [142]:
point = data

In [143]:
transp = data.transpose()
transp.columns = transp.iloc[0]
transp.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Eagleville city, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","Smyrna town, Tennessee","Watertown city, Tennessee",Tennessee
NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Eagleville city, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","Smyrna town, Tennessee","Watertown city, Tennessee",Tennessee
GEO_ID,0500000US47161,0500000US47125,0500000US47083,0500000US47085,0500000US47043,0500000US47021,0500000US47147,0500000US47165,0500000US47037,0500000US47189,0500000US47169,0500000US47187,0500000US47149,0500000US47119,1600000US4722360,1600000US4741200,1600000US4741520,1600000US4750780,1600000US4751560,1600000US4769420,1600000US4778320,0400000US47
agebysex_total_series,13553,204992,8201,18528,53289,40539,70982,187680,690540,140604,10910,232380,324139,94615,943,35556,34759,35834,141704,50820,1698,6772268
age_total_male,6572,102205,4119,9113,26246,20221,35176,91606,332527,68930,6668,113906,159299,45700,472,17716,16515,17691,68891,25143,732,3304462
age_m_u5,353,8916,216,538,1733,1192,2325,5776,23406,4318,373,6932,10772,3200,48,1655,1205,1252,4640,1848,63,208551


In [144]:
transp = transp.loc[transp['Stewart County, Tennessee'] != 'Stewart County, Tennessee']
transp = transp.loc[transp['Stewart County, Tennessee'] != '0500000US47161']

In [145]:
transp.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee","Eagleville city, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","Mount Juliet city, Tennessee","Murfreesboro city, Tennessee","Smyrna town, Tennessee","Watertown city, Tennessee",Tennessee
agebysex_total_series,13553,204992,8201,18528,53289,40539,70982,187680,690540,140604,10910,232380,324139,94615,943,35556,34759,35834,141704,50820,1698,6772268
age_total_male,6572,102205,4119,9113,26246,20221,35176,91606,332527,68930,6668,113906,159299,45700,472,17716,16515,17691,68891,25143,732,3304462
age_m_u5,353,8916,216,538,1733,1192,2325,5776,23406,4318,373,6932,10772,3200,48,1655,1205,1252,4640,1848,63,208551
age_m_5to9,476,7959,347,565,1662,1165,2352,6523,18746,4573,425,9230,11045,2504,60,1064,1235,1373,4664,2435,35,208045
age_m_10to14,364,7229,234,663,1850,1407,2491,6312,20115,5100,268,9690,12034,3692,40,1661,1129,1699,4978,1688,67,222622


In [146]:
numcols = list(transp.columns)
numcols
transp[numcols] = transp[numcols].astype(float)

In [147]:
data = transp

In [148]:
data['GNRC'] = data['Stewart County, Tennessee']+data['Montgomery County, Tennessee']+data['Houston County, Tennessee']+data['Humphreys County, Tennessee']+data['Dickson County, Tennessee']+data['Cheatham County, Tennessee']+data['Robertson County, Tennessee']+data['Sumner County, Tennessee']+data['Davidson County, Tennessee']+data['Wilson County, Tennessee']+data['Trousdale County, Tennessee']+data['Williamson County, Tennessee']+data['Rutherford County, Tennessee']

In [149]:
data['MPO'] = data['Robertson County, Tennessee']+data['Sumner County, Tennessee']+data['Davidson County, Tennessee']+data['Wilson County, Tennessee']+data['Williamson County, Tennessee']+data['Rutherford County, Tennessee']+data['Maury County, Tennessee']

In [151]:
data['Rutherford Incorporated']=data['Eagleville city, Tennessee']+data['La Vergne city, Tennessee']+data['Murfreesboro city, Tennessee']+data['Smyrna town, Tennessee']

In [152]:
data['Rutherford Unincorporated'] = data['Rutherford County, Tennessee'] - data['Rutherford Incorporated']

In [153]:
data['Wilson Incorporated'] = data['Lebanon city, Tennessee']+data['Mount Juliet city, Tennessee']+data['Watertown city, Tennessee']
data['Wilson Unincorporated'] = data['Wilson County, Tennessee'] - data['Wilson Incorporated']

In [155]:
data = data.drop(columns = ['Lebanon city, Tennessee','Mount Juliet city, Tennessee','Watertown city, Tennessee','Eagleville city, Tennessee','La Vergne city, Tennessee','Murfreesboro city, Tennessee','Smyrna town, Tennessee'])

In [156]:
data.head()

NAME,"Stewart County, Tennessee","Montgomery County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Dickson County, Tennessee","Cheatham County, Tennessee","Robertson County, Tennessee","Sumner County, Tennessee","Davidson County, Tennessee","Wilson County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Rutherford County, Tennessee","Maury County, Tennessee",Tennessee,GNRC,MPO,Rutherford Incorporated,Rutherford Unincorporated,Wilson Incorporated,Wilson Unincorporated
agebysex_total_series,13553.0,204992.0,8201.0,18528.0,53289.0,40539.0,70982.0,187680.0,690540.0,140604.0,10910.0,232380.0,324139.0,94615.0,6772268.0,1996337.0,1740940.0,229023.0,95116.0,72291.0,68313.0
age_total_male,6572.0,102205.0,4119.0,9113.0,26246.0,20221.0,35176.0,91606.0,332527.0,68930.0,6668.0,113906.0,159299.0,45700.0,3304462.0,976588.0,847144.0,112222.0,47077.0,34938.0,33992.0
age_m_u5,353.0,8916.0,216.0,538.0,1733.0,1192.0,2325.0,5776.0,23406.0,4318.0,373.0,6932.0,10772.0,3200.0,208551.0,66850.0,56729.0,8191.0,2581.0,2520.0,1798.0
age_m_5to9,476.0,7959.0,347.0,565.0,1662.0,1165.0,2352.0,6523.0,18746.0,4573.0,425.0,9230.0,11045.0,2504.0,208045.0,65068.0,54973.0,8223.0,2822.0,2643.0,1930.0
age_m_10to14,364.0,7229.0,234.0,663.0,1850.0,1407.0,2491.0,6312.0,20115.0,5100.0,268.0,9690.0,12034.0,3692.0,222622.0,67757.0,59434.0,8367.0,3667.0,2895.0,2205.0


In [157]:
ok = data.transpose().reset_index()

In [158]:
ok.head()

,NAME,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,healthcoverage_m55to64_wprivate,healthcoverage_m55to64_woprivate,healthcoverage_m65to74_privateall,healthcoverage_m65to74_wprivate,healthcoverage_m65to74_woprivate,healthcoverage_m75+_privateall,healthcoverage_m75+_wprivate,healthcoverage_m75+_woprivate,healthcoverage_total_fprivate,healthcoverage_fu6_privateall,healthcoverage_fu6_wprivate,healthcoverage_fu6_woprivate,healthcoverage_f6to18_privateall,healthcoverage_f6to18_wprivate,healthcoverage_f6to18_woprivate,healthcoverage_f19to25_privateall,healthcoverage_f19to25_wprivate,healthcoverage_f19to25_woprivate,healthcoverage_f26to34_privateall,healthcoverage_f26to34_wprivate,healthcoverage_f26to34_woprivate,healthcoverage_f35to44_privateall,healthcoverage_f35to44_wprivate,healthcoverage_f35to44_woprivate,healthcoverage_f45to54_privateall,healthcoverage_f45to54_wprivate,healthcoverage_f45to54_woprivate,healthcoverage_f55to64_privateall,healthcoverage_f55to64_wpr

In [159]:
ok.tail()

,NAME,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,healthcoverage_total_healthcare_series,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_mu6_w,healthcoverage_mu6_wo,healthcoverage_m6to18_all,healthcoverage_m6to18_w,healthcoverage_m6to18_wo,healthcoverage_m19to25_all,healthcoverage_m19to25_w,healthcoverage_m19to25_wo,healthcoverage_m26to34_all,healthcoverage_m26to34_w,healthcoverage_m26to34_wo,healthcoverage_m35to44_all,healthcoverage_m35to44_w,healthcoverage_m35to44_wo,healthcoverage_m45to54_all,healthcoverage_m45to54_w,healthcoverage_m45to54_wo,healthcoverage_m55to64_all,healthcoverage_m55to64_w,healthcoverage_m55to64_wo,healthcoverage_m65to74_all,healthcoverage_m65to74_w,healthcoverage_m65to74_wo,healthcoverage_m75+_all,healthcoverage_m75+_w,healthcoverage_m75+_wo,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_fu6_w,healthcoverage_fu6_wo,healthcoverage_f6to18_all,healthcoverage_f6to18_w,healthcoverage_f6to18_wo,healthcoverage_f19to25_all,healthcoverage_f19to25_w,healthcoverage_f19to25_wo,healthcoverage_f26to34_all,healthcoverage_f26to34_w,healthcoverage_f26to34_wo,healthcoverage_f35to44_all,healthcoverage_f35to44_w,healthcoverage_f35to44_wo,healthcoverage_f45to54_all,healthcoverage_f45to54_w,healthcoverage_f45to54_wo,healthcoverage_f55to64_all,healthcoverage_f55to64_w,healthcoverage_f55to64_wo,healthcoverage_f65to74_all,healthcoverage_f65to74_w,healthcoverage_f65to74_wo,healthcoverage_f75+_all,healthcoverage_f75+_w,healthcoverage_f75+_wo,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_wprivate,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_wprivate,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_wprivate,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_wprivate,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_wprivate,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_wprivate,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,healthcoverage_m55to64_wprivate,healthcoverage_m55to64_woprivate,healthcoverage_m65to74_privateall,healthcoverage_m65to74_wprivate,healthcoverage_m65to74_woprivate,healthcoverage_m75+_privateall,healthcoverage_m75+_wprivate,healthcoverage_m75+_woprivate,healthcoverage_total_fprivate,healthcoverage_fu6_privateall,healthcoverage_fu6_wprivate,healthcoverage_fu6_woprivate,healthcoverage_f6to18_privateall,healthcoverage_f6to18_wprivate,healthcoverage_f6to18_woprivate,healthcoverage_f19to25_privateall,healthcoverage_f19to25_wprivate,healthcoverage_f19to25_woprivate,healthcoverage_f26to34_privateall,healthcoverage_f26to34_wprivate,healthcoverage_f26to34_woprivate,healthcoverage_f35to44_privateall,healthcoverage_f35to44_wprivate,healthcoverage_f35to44_woprivate,healthcoverage_f45to54_privateall,healthcoverage_f45to54_wprivate,healthcoverage_f45to54_woprivate,healthcoverage_f55to64_privateall,healthcoverage_f55to64_wpr

In [160]:
ok.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Columns: 785 entries, NAME to GeoFIPS
dtypes: float64(784), object(1)
memory usage: 128.9+ KB


In [161]:
ok.to_csv('../data/ACS20205YR.csv', index = False)